# Eksperimen 11.6: Perbandingan Model DNN dan LeNet-5 pada Dataset MNIST

Eksperimen ini bertujuan untuk membangun, melatih, dan membandingkan dua arsitektur *Deep Learning* dalam melakukan tugas klasifikasi gambar angka tulisan tangan dari dataset **MNIST**:

1. **Persiapan dan Normalisasi Data**: 
   Dataset MNIST diunduh dan dipisahkan menjadi *Training Set* dan *Test Set*. Nilai piksel gambar kemudian dinormalisasi (dibagi 255.0) agar berada pada rentang nilai 0 hingga 1. Normalisasi ini sangat penting agar proses pelatihan model berjalan lebih stabil dan konvergen lebih cepat.

2. **Skenario 1: Deep Neural Network (DNN)**: 
   Model pertama menggunakan jaringan saraf tiruan standar (*Fully-Connected Network*) yang dipipihkan (*Flatten*). Model ini memiliki 2 *hidden layers* dengan masing-masing 128 neuron berspesifikasi aktivasi ReLU. Untuk mencegah terjadinya *overfitting*, diterapkan *Regulation Dropout* sebesar 30% pada setiap *layer*. Model DNN ini memiliki total parameter yang cukup besar (sekitar 118,282 parameter).

3. **Skenario 2: Convolutional Neural Network (Arsitektur LeNet-5)**: 
   Model kedua mengadaptasi arsitektur klasik LeNet-5 yang memang didesain khusus untuk mengekstraksi fitur spasial pada data citra. Model ini menggunakan kombinasi *Convolutional Layer* (`Conv2D`) dengan fungsi aktivasi `tanh` dan *Pooling Layer* (`AveragePooling2D`). Berbeda dengan DNN, LeNet-5 memiliki jumlah parameter jaringan yang jauh lebih efisien (sekitar 61,706 parameter).

**Tujuan Evaluasi**: 
Eksperimen ini akan mendemonstrasikan bahwa meskipun arsitektur **LeNet-5 (CNN)** memiliki jumlah parameter yang jauh lebih sedikit dibandingkan model DNN standar, ia mampu menangkap pola gambar dua dimensi dengan lebih baik, sehingga pada akhirnya menghasilkan akurasi evaluasi yang lebih tinggi pada *Test Set*.

In [6]:
import numpy as np
from tensorflow.keras import datasets
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dropout, Dense
from tensorflow.keras.layers import AveragePooling2D, Conv2D

# ---------------------------------------------------------
# PERSIAPAN DATA UMUM
# ---------------------------------------------------------
# Unduh dan siapkan dataset MNIST
mnist = datasets.mnist
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Lakukan normalisasi nilai piksel gambar menjadi 0 – 1:
X_train = X_train/255.0
X_test = X_test/255.0

# ---------------------------------------------------------
# SKENARIO 1: Model DNN dengan 2 hidden layers
# ---------------------------------------------------------
model4 = Sequential()
model4.add(Flatten(input_shape=[28, 28]))
model4.add(Dropout(0.3))
model4.add(Dense(128, activation='relu'))
model4.add(Dropout(0.3))
model4.add(Dense(128, activation='relu'))
model4.add(Dropout(0.3))
model4.add(Dense(10, activation='softmax'))

# Gunakan Adam sebagai optimizer dan kompilasi model
# (Catatan: tanda kutip pada 'Adam' disesuaikan ke standar string Python)
model4.compile(loss='sparse_categorical_crossentropy',
optimizer='Adam', metrics=['accuracy'])

# Jalankan eksperimen sampai 50 epochs
model4.fit(X_train, y_train, epochs=50,
validation_split=0.2,
batch_size=128)

# Evaluasi model dengan menggunakan Test Set:
model4.evaluate(X_test, y_test)

# ---------------------------------------------------------
# PERSIAPAN DATA KHUSUS UNTUK LeNet-5
# ---------------------------------------------------------
# LeNet-5 butuh input (32, 32, 1), sedangkan MNIST (28, 28). Kita tambahkan padding.
X_train_pad = np.pad(X_train, ((0,0), (2,2), (2,2)), mode='constant')
X_test_pad = np.pad(X_test, ((0,0), (2,2), (2,2)), mode='constant')

# Tambahkan dimensi channel di akhir agar shape menjadi (32, 32, 1)
X_train_lenet = np.expand_dims(X_train_pad, axis=-1)
X_test_lenet = np.expand_dims(X_test_pad, axis=-1)

# ---------------------------------------------------------
# SKENARIO 2: LeNet-5
# ---------------------------------------------------------
# Siapkan model CNN LeNet-5 tanpa Dropout layer (KODE ASLI TIDAK DIUBAH)
model5 = Sequential()
model5.add(Conv2D(filters=6, kernel_size=(5,5),
activation='tanh', input_shape=(32,32,1)))
model5.add(AveragePooling2D(pool_size =(2, 2), strides =(2, 2)))
model5.add(Conv2D(filters=16, kernel_size=(5,5),
activation='tanh'))
model5.add(AveragePooling2D(pool_size =(2, 2), strides =(2, 2)))
model5.add(Flatten())
model5.add(Dense(units=120, activation='tanh'))
model5.add(Dense(units=84, activation='tanh'))
model5.add(Dense(units=10, activation = 'softmax'))

# Gunakan Adam sebagai optimizer dan kompilasi model
model5.compile(loss='sparse_categorical_crossentropy',
optimizer='Adam', metrics=['accuracy'])

# Jalankan eksperimen sampai epochs 50 (MENGGUNAKAN DATA YANG SUDAH DI-PADDING)
model5.fit(X_train_lenet, y_train, epochs=50,
validation_split=0.2,
batch_size=128)

# Evaluasi model dengan menggunakan Test Set (MENGGUNAKAN DATA YANG SUDAH DI-PADDING)
model5.evaluate(X_test_lenet, y_test)

Epoch 1/50
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7968 - loss: 0.6409 - val_accuracy: 0.9377 - val_loss: 0.2144
Epoch 2/50
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9014 - loss: 0.3219 - val_accuracy: 0.9534 - val_loss: 0.1555
Epoch 3/50
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9205 - loss: 0.2620 - val_accuracy: 0.9609 - val_loss: 0.1309
Epoch 4/50
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9311 - loss: 0.2253 - val_accuracy: 0.9656 - val_loss: 0.1124
Epoch 5/50
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9389 - loss: 0.2004 - val_accuracy: 0.9687 - val_loss: 0.1055
Epoch 6/50
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9429 - loss: 0.1873 - val_accuracy: 0.9703 - val_loss: 0.0983
Epoch 7/50
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9466 - loss: 0.1729 - val_accuracy: 0.9683 - val_loss: 0.0977
Epoch 8/50
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9479 - loss: 0.1669 - val_accuracy: 0.

[0.04738345369696617, 0.9894000291824341]